In [3]:
import sqlite3
import unicodedata
import pandas as pd
from pathlib import Path

In [5]:
carpeta = Path("../datos_brutos")
bd = sqlite3.connect("../sql/alzina.db")

In [6]:
def nombre_columna(c):
    # "Nº Factura" -> "n_factura", "Fecha emisión" -> "fecha_emision"
    c = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
    c = c.lower().strip().replace("%", "pct").replace("/", "_")
    return "_".join("".join(ch if ch.isalnum() else " " for ch in c).split())

ficheros = {
    "bruto_facturas_antiguas": ("facturas_emitidas_2024_abr2025.csv", dict(sep=";", encoding="cp1252")),
    "bruto_facturas_nuevas":   ("facturas_emitidas_may2025_sep2026.xlsx", {}),
    "bruto_obras":             ("control_obras.xlsx", {}),
    "bruto_gastos":            ("gastos_gestoria.xlsx", {}),
    "bruto_nominas":           ("nominas_resumen.xlsx", {}),
    "bruto_partes":            ("partes_horas.csv", {}),
    "bruto_banco":             ("movimientos_banco.csv", dict(sep=";", encoding="utf-8-sig")),
}

for tabla, (fichero, opciones) in ficheros.items():
    ruta = carpeta / fichero
    if ruta.suffix == ".xlsx":
        df = pd.read_excel(ruta, dtype=str)
    else:
        df = pd.read_csv(ruta, dtype=str, **opciones)
    df.columns = [nombre_columna(c) for c in df.columns]
    df.to_sql(tabla, bd, if_exists="replace", index=False)
    print(f"{tabla:28} {len(df):>5} filas  ->  {list(df.columns)}")

bd.close()

bruto_facturas_antiguas        159 filas  ->  ['no_factura', 'fecha', 'cliente', 'nif_cif', 'concepto', 'base_imponible', 'pct_iva', 'cuota_iva', 'total', 'forma_de_pago', 'vencimiento', 'factura_rectificada']
bruto_facturas_nuevas          191 filas  ->  ['numero', 'fecha_emision', 'cliente', 'cif_nif', 'descripcion', 'base', 'iva_pct', 'importe_iva', 'total', 'fecha_vencimiento', 'estado', 'rectifica_a']
bruto_obras                    224 filas  ->  ['codigo', 'cliente', 'poblacion', 'tipo_trabajo', 'fecha_inicio', 'fecha_fin', 'presupuesto', 'observaciones']
bruto_gastos                  1234 filas  ->  ['fecha_factura', 'no_factura', 'proveedor', 'nif_proveedor', 'concepto', 'categoria', 'obra', 'base', 'pct_iva', 'cuota_iva', 'retencion_irpf', 'total_a_pagar']
bruto_nominas                  228 filas  ->  ['mes', 'trabajador', 'categoria', 'concepto', 'devengado', 'ss_empresa', 'ss_trabajador', 'irpf', 'liquido']
bruto_partes                  1754 filas  ->  ['semana_inicio', 'ope

Lee cada fichero todo como texto (dtype=str), para no cambiar nada. Los arreglos los harás tú en SQL.
Cambia los nombres de las columnas a un formato cómodo para SQL: sin espacios, tildes ni símbolos.
Guarda cada fichero en una tabla que empieza por bruto_, dentro de sql/alzina.db.